___
# <center>Da amostra para a população</center>
___

## Aula 09

**Objetivo da aula:** ao final desta aula, você deve ser capaz de:

 * separar parâmetro, estimador e estimativa num enunciado;
 * simular o que acontece quando se sorteia uma amostra muitas vezes;
 * reconhecer o teorema central do limite no resultado da simulação;
 * calcular e interpretar um intervalo de confiança sem dizer o que ele não diz.

Este notebook faz uma coisa que a lousa não faz: repete o sorteio dez mil vezes.
É essa repetição que torna visível a ideia central da aula.


___
<div id="indice"></div>

## Índice

- [Três palavras que não são sinônimos](#palavras)

- [O sorteio, uma vez](#uma)

- [O sorteio, dez mil vezes](#muitas)

- [O intervalo de confiança](#intervalo)

- [O que o intervalo NÃO diz](#cuidado)

- [RESUMO](#resumo)


___
<div id="palavras"></div>

# Três palavras que não são sinônimos

| palavra | o que é | neste caso |
|---|---|---|
| **parâmetro** | o número da população, fixo e desconhecido | a proporção de regime fechado em todos os acórdãos do TJSP |
| **estimador** | a receita de cálculo | "a proporção na amostra" |
| **estimativa** | o número que saiu desta amostra | 0,456 |

A estatística inteira é a tentativa de falar do primeiro tendo só o terceiro.


In [ ]:
import pandas as pd

pd.set_option("display.max_columns", 30)
pd.set_option("display.width", 160)

URL = "https://raw.githubusercontent.com/jtrecenti/202662-cdad2/main/dados"

criminal = pd.read_csv(f"{URL}/tjsp_cjsg_criminal.csv")

# Fora as linhas sem regime: não dá para calcular proporção de regime
# em acórdão que não informou regime nenhum.
penas = criminal.dropna(subset=["regime_inicial"])

penas.shape


Para hoje vamos fingir que a nossa base de 333 acórdãos **é** a população
inteira. Assim conhecemos o parâmetro, o que na vida real nunca acontece, e
podemos conferir se o método funciona.


In [ ]:
import numpy as np
from plotnine import *

fechado = penas["regime_inicial"] == "fechado"

# O parâmetro. Na vida real este número não existe para você.
parametro = fechado.mean()

round(parametro, 4)


[Volta ao Índice](#indice)


___
<div id="uma"></div>

# O sorteio, uma vez

Sorteamos 50 acórdãos e calculamos a proporção só neles.


In [ ]:
amostra = penas.sample(50, random_state=1)

(amostra["regime_inicial"] == "fechado").mean()


**✍️ Agora você.** Troque o `random_state` por outro número e rode de novo. Depois mais uma vez. O que acontece com a estimativa?


In [ ]:
amostra = penas.sample(50, random_state=________)

(amostra["regime_inicial"] == "fechado").mean()


Cada sorteio dá um número diferente, e nenhum deles é o parâmetro. **A
estimativa é uma variável aleatória**: ela tem distribuição, como tudo que
vimos na quinta-feira.


[Volta ao Índice](#indice)


___
<div id="muitas"></div>

# O sorteio, dez mil vezes

Se a estimativa tem distribuição, vamos olhar para ela. Sorteamos dez mil
amostras de 50 e guardamos a proporção de cada uma.


In [ ]:
estimativas = [
    (penas.sample(50, random_state=i)["regime_inicial"] == "fechado").mean()
    for i in range(10_000)
]

simulacao = pd.DataFrame({"estimativa": estimativas})

simulacao["estimativa"].describe().round(4)


In [ ]:
(
    ggplot(simulacao)
    + aes(x="estimativa")
    + geom_histogram(bins=30, fill="#3ACC9F", color="white")
    + geom_vline(xintercept=parametro, color="#E50505", size=1.2)
    + labs(x="proporção de regime fechado em amostras de 50",
           y="quantas amostras",
           title="A distribuição da estimativa (a linha vermelha é o parâmetro)")
    + theme_minimal()
)


Três coisas para reparar, e são as três ideias da aula:

1. o monte está **centrado no parâmetro**: em média a estimativa acerta;
2. o formato é de **sino**, mesmo a variável original sendo só sim ou não. Isso
   é o teorema central do limite;
3. a **largura** do monte é o erro que se corre ao usar uma amostra só.


**✍️ Agora você.** Refaça com amostras de 200 em vez de 50 e compare o desvio-padrão das estimativas. Quadruplicar a amostra divide a largura por quanto?


In [ ]:
maiores = [
    (penas.sample(________, random_state=i)["regime_inicial"] == "fechado").mean()
    for i in range(10_000)
]

print("com  50:", round(np.std(estimativas), 4))
print("com 200:", round(np.std(________), 4))


[Volta ao Índice](#indice)


___
<div id="intervalo"></div>

# O intervalo de confiança

Na vida real você tem **uma** amostra, e não dez mil. O intervalo de confiança
é o jeito de carregar a largura daquele monte junto com a estimativa.


In [ ]:
amostra = penas.sample(50, random_state=1)
p_chapeu = (amostra["regime_inicial"] == "fechado").mean()
n = len(amostra)

# O erro-padrão: a largura do monte, estimada a partir da própria amostra
erro_padrao = np.sqrt(p_chapeu * (1 - p_chapeu) / n)

# 1,96 é o que cobre 95% de uma normal
margem = 1.96 * erro_padrao

print("estimativa:", round(p_chapeu, 3))
print("intervalo :", round(p_chapeu - margem, 3), "a", round(p_chapeu + margem, 3))
print("parâmetro :", round(parametro, 3))


Repare no $\sqrt{n}$ na conta do erro-padrão. É dele que vem a regra da aula:
para estreitar o intervalo pela metade, é preciso **quatro vezes** mais
amostra. É por isso que pesquisa eleitoral para de crescer em 2.000
entrevistados.


[Volta ao Índice](#indice)


___
<div id="cuidado"></div>

# O que o intervalo NÃO diz

A leitura errada é dizer que "há 95% de chance de o parâmetro estar neste
intervalo". O parâmetro é fixo: ou está, ou não está.

Os 95% são uma propriedade do **método**. Vamos verificar isso construindo mil
intervalos e contando quantos pegaram o parâmetro.


In [ ]:
pegou = 0
for i in range(1_000):
    a = penas.sample(50, random_state=i)
    p = (a["regime_inicial"] == "fechado").mean()
    m = 1.96 * np.sqrt(p * (1 - p) / len(a))
    if p - m <= parametro <= p + m:
        pegou += 1

print(f"{pegou} de 1.000 intervalos contêm o parâmetro "
      f"({pegou / 10:.1f}%)")


Perto de 95%. **Esse** é o sentido da confiança: se você repetisse o
procedimento a vida inteira, erraria em cerca de 5% das vezes.

Da amostra que você tem na mão, você não sabe se ela é uma das 95 ou uma das 5.


<div id="ex1"></div>

### EXERCÍCIO 1

Um relatório afirma: *"analisamos 400 sentenças e 62% foram procedentes, com
intervalo de confiança de 57% a 67%"*. Escreva, em uma frase cada:

1. qual é o parâmetro, qual é a estimativa;
2. uma leitura **correta** do intervalo, para colocar no relatório;
3. o que muda se as 400 sentenças não tiverem sido sorteadas, e sim escolhidas
   entre as que o escritório já tinha em pasta.


💡 A terceira pergunta é a mais importante do dia. Todo este notebook depende de
`.sample()`, que sorteia. Amostra que não foi sorteada não tem intervalo de
confiança que a salve: o erro deixa de ser aleatório e vira viés, e viés não
diminui com mais dados.


[Volta ao Índice](#indice)


___
<div id="resumo"></div>

# RESUMO

| ideia | o que fizemos |
|---|---|
| parâmetro | o número da população, fixo e desconhecido |
| estimativa | o que saiu desta amostra, e que muda a cada sorteio |
| distribuição da estimativa | dez mil sorteios, e o histograma deles |
| teorema central do limite | o histograma vira sino, mesmo com variável de sim ou não |
| erro-padrão | $\sqrt{p(1-p)/n}$: a largura daquele histograma |
| intervalo de 95% | estimativa $\pm$ 1,96 erros-padrão |

**A frase para levar:** o intervalo de confiança é uma declaração sobre o
método, não sobre este intervalo. E ele só vale se a amostra foi sorteada.


[Volta ao Índice](#indice)
